# Session 5: Introduction to Switching and VLANs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-05-switching-and-vlans.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-05_switch_sim.py`

In [1]:
"""Session 5 - How a Layer 2 switch learns, floods, forwards and filters, per VLAN.

The switch keeps one MAC address table. Each entry is (VLAN, MAC) -> port.
- Learn: record the SOURCE MAC of every frame against the port it arrived on.
- Forward: if the DESTINATION is known in that VLAN, send it out that one port.
- Filter: if the destination is on the same port it came from, drop it.
- Flood: unknown unicast and broadcast go out every other port IN THE SAME VLAN.
Run:  python session-05_switch_sim.py
"""

PORTS = {"Fa0/1": 10, "Fa0/2": 10, "Fa0/3": 20, "Fa0/4": 20, "Fa0/5": 10}   # access ports
HOSTS = {"PC-A": ("aaaa.aaaa.0001", "Fa0/1"), "PC-B": ("bbbb.bbbb.0002", "Fa0/2"),
         "PC-C": ("cccc.cccc.0003", "Fa0/3"), "PC-D": ("dddd.dddd.0004", "Fa0/4"),
         "PRN-E": ("eeee.eeee.0005", "Fa0/5")}
BROADCAST = "ffff.ffff.ffff"


class Switch:
    def __init__(self, ports):
        self.ports = ports
        self.table = {}

    def receive(self, in_port, src, dst):
        vlan = self.ports[in_port]
        learned = (vlan, src) not in self.table
        self.table[(vlan, src)] = in_port
        if dst == BROADCAST or (vlan, dst) not in self.table:
            out = [p for p, v in self.ports.items() if v == vlan and p != in_port]
            action = "FLOOD" if dst != BROADCAST else "BROADCAST"
        elif self.table[(vlan, dst)] == in_port:
            out, action = [], "FILTER"
        else:
            out, action = [self.table[(vlan, dst)]], "FORWARD"
        return vlan, learned, action, out

    def show_mac_table(self):
        print("          Mac Address Table")
        print("Vlan    Mac Address       Type        Ports")
        print("----    -----------       --------    -----")
        for (vlan, mac), port in sorted(self.table.items()):
            print(f"{vlan:<7} {mac:17} DYNAMIC     {port}")


if __name__ == "__main__":
    sw = Switch(PORTS)
    events = [("PC-A", "PC-B"), ("PC-B", "PC-A"), ("PC-C", "BCAST"), ("PC-A", "PRN-E"),
              ("PRN-E", "PC-A"), ("PC-A", "PC-C"), ("PC-D", "PC-C")]
    for s, d in events:
        smac, sport = HOSTS[s]
        dmac = BROADCAST if d == "BCAST" else HOSTS[d][0]
        vlan, learned, action, out = sw.receive(sport, smac, dmac)
        note = " (learned new MAC)" if learned else ""
        print(f"{s:5} -> {d:5} in {sport} VLAN {vlan}: {action:9} out {', '.join(out) or '-'}{note}")
    print()
    sw.show_mac_table()
    print("\nPC-A -> PC-C never reaches VLAN 20: VLANs are separate broadcast domains.")
    print("They need a router or Layer 3 switch (Session 12) to talk.")

PC-A  -> PC-B  in Fa0/1 VLAN 10: FLOOD     out Fa0/2, Fa0/5 (learned new MAC)
PC-B  -> PC-A  in Fa0/2 VLAN 10: FORWARD   out Fa0/1 (learned new MAC)
PC-C  -> BCAST in Fa0/3 VLAN 20: BROADCAST out Fa0/4 (learned new MAC)
PC-A  -> PRN-E in Fa0/1 VLAN 10: FLOOD     out Fa0/2, Fa0/5
PRN-E -> PC-A  in Fa0/5 VLAN 10: FORWARD   out Fa0/1 (learned new MAC)
PC-A  -> PC-C  in Fa0/1 VLAN 10: FLOOD     out Fa0/2, Fa0/5
PC-D  -> PC-C  in Fa0/4 VLAN 20: FORWARD   out Fa0/3 (learned new MAC)

          Mac Address Table
Vlan    Mac Address       Type        Ports
----    -----------       --------    -----
10      aaaa.aaaa.0001    DYNAMIC     Fa0/1
10      bbbb.bbbb.0002    DYNAMIC     Fa0/2
10      eeee.eeee.0005    DYNAMIC     Fa0/5
20      cccc.cccc.0003    DYNAMIC     Fa0/3
20      dddd.dddd.0004    DYNAMIC     Fa0/4

PC-A -> PC-C never reaches VLAN 20: VLANs are separate broadcast domains.
They need a router or Layer 3 switch (Session 12) to talk.


## `session-05_dtp_matrix.py`

In [2]:
"""Session 5 - Will this link become a trunk? The DTP negotiation matrix.

Dynamic Trunking Protocol (Cisco) lets two switch ports agree on trunk or
access. Security best practice is to switch it off: hard-code access ports
with "switchport mode access" + "switchport nonegotiate", and trunks with
"switchport mode trunk" + "switchport nonegotiate".
Run:  python session-05_dtp_matrix.py
"""

MODES = ["access", "dynamic auto", "dynamic desirable", "trunk"]


def result(a, b):
    if "access" in (a, b):
        if {a, b} == {"access", "trunk"}:
            return "mismatch!"         # one side tags, the other does not
        return "access"
    if a == "trunk" or b == "trunk":
        return "trunk"
    if "dynamic desirable" in (a, b):
        return "trunk"
    return "access"                      # auto + auto: nobody asks


if __name__ == "__main__":
    w = 18
    print(" " * w + "".join(f"{m:>{w}}" for m in MODES))
    for a in MODES:
        print(f"{a:<{w}}" + "".join(f"{result(a, b):>{w}}" for b in MODES))
    print("\nWhy it matters for security:")
    print(" - A port left in 'dynamic desirable' or 'dynamic auto' can be talked into trunking")
    print("   by a device that speaks DTP, giving it access to every VLAN (VLAN hopping).")
    print(" - Fix: switchport mode access + switchport nonegotiate on every user port.")
    print("\nVTP reminder: a switch in server mode with a HIGHER revision number")
    print("overwrites the VLAN database of every client in the domain. Use VTP transparent")
    print("or VTP v3 with a primary server, and reset the revision before adding a switch.")

                              access      dynamic auto dynamic desirable             trunk
access                        access            access            access         mismatch!
dynamic auto                  access            access             trunk             trunk
dynamic desirable             access             trunk             trunk             trunk
trunk                      mismatch!             trunk             trunk             trunk

Why it matters for security:
 - A port left in 'dynamic desirable' or 'dynamic auto' can be talked into trunking
   by a device that speaks DTP, giving it access to every VLAN (VLAN hopping).
 - Fix: switchport mode access + switchport nonegotiate on every user port.

VTP reminder: a switch in server mode with a HIGHER revision number
overwrites the VLAN database of every client in the domain. Use VTP transparent
or VTP v3 with a primary server, and reset the revision before adding a switch.
